# EEGTCNet: k15_s4 + Pretrain

- Architecture: EEGTCNet (k=15, stride=4, 151 tokens)
- Self-supervised pretraining: random 15% mask, 50 epochs
- 64 channels, 160 Hz, bandpass 7-30 Hz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

import mne
mne.set_log_level('WARNING')

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def set_random_seeds(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_random_seeds(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Dataset
N_SUBJECTS = 109
N_CHANNELS = 64
SFREQ = 160
N_CLASSES = 2
EPOCH_SAMPLES = 641
MI_ME_RUNS = [3, 4, 7, 8, 11, 12]
BANDPASS_LOW = 7.0
BANDPASS_HIGH = 30.0

# Model
POOL_KERNEL = 15
POOL_STRIDE = 4
EMB_SIZE = 40
TF_DIM = 64
NUM_TF_LAYERS = 3
NUM_HEADS = 4

# Training
LR = 3e-4
BATCH_SIZE = 72
MAX_EPOCHS = 150
PATIENCE = 10
WEIGHT_DECAY = 1e-4

# Pretraining
PRETRAIN_EPOCHS = 50
PRETRAIN_LR = 1e-4
PRETRAIN_MASK_RATIO = 0.15

# Fine-tuning
CALIBRATION_FRACTIONS = [0.05, 0.10, 0.20, 0.50]
FT_LR = 1e-3
FT_EPOCHS = 10

# Paths
DATA_DIR = 'eeg-models/physionet_data'
SAVE_DIR = 'eeg-models/saved_models_v18'

PyTorch 2.8.0+cu128, Device: cuda
GPU: NVIDIA A100 80GB PCIe


## Data Loading

In [2]:
def load_physionet_subject(subject_id, data_dir=DATA_DIR):
    subject_name = f'S{subject_id:03d}'
    all_epochs, all_labels = [], []
    for run in MI_ME_RUNS:
        edf_path = os.path.join(data_dir, f'{subject_name}R{run:02d}.edf')
        if not os.path.exists(edf_path):
            edf_path = os.path.join(data_dir, subject_name, f'{subject_name}R{run:02d}.edf')
            if not os.path.exists(edf_path):
                continue
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        if raw.info['sfreq'] != SFREQ:
            raw = raw.copy().resample(SFREQ, verbose=False)
        rename_map = {ch: ch.rstrip('.') for ch in raw.info['ch_names']}
        raw.rename_channels(rename_map)
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        mi_event_id = {k: v for k, v in event_id.items() if k in ('T1', 'T2')}
        if len(mi_event_id) < 2:
            continue
        epochs = mne.Epochs(raw, events, event_id=mi_event_id,
                            tmin=0.0, tmax=4.0, baseline=None,
                            preload=True, verbose=False)
        data = epochs.get_data()
        if data.shape[2] > EPOCH_SAMPLES:
            data = data[:, :, :EPOCH_SAMPLES]
        t1_val, t2_val = mi_event_id['T1'], mi_event_id['T2']
        for i, ev in enumerate(epochs.events[:, 2]):
            if ev == t1_val:
                all_labels.append(0)
                all_epochs.append(data[i])
            elif ev == t2_val:
                all_labels.append(1)
                all_epochs.append(data[i])
    if not all_epochs:
        return None
    return np.array(all_epochs), np.array(all_labels)


def load_all_subjects(data_dir=DATA_DIR, n_subjects=N_SUBJECTS):
    subjects_data = {}
    for sid in range(1, n_subjects + 1):
        result = load_physionet_subject(sid, data_dir=data_dir)
        if result is not None:
            X, y = result
            subjects_data[sid] = (X, y)
            print(f'  S{sid:03d}: {len(X)} epochs (L={np.sum(y==0)}, R={np.sum(y==1)})')
    return subjects_data


print('Loading PhysioNet data...')
subjects_data = load_all_subjects()
print(f'Loaded {len(subjects_data)} subjects, {sum(len(y) for _, y in subjects_data.values())} total epochs')

Loading PhysioNet data...
  S001: 90 epochs (L=46, R=44)
  S002: 90 epochs (L=46, R=44)
  S003: 90 epochs (L=45, R=45)
  S004: 90 epochs (L=45, R=45)
  S005: 90 epochs (L=44, R=46)
  S006: 90 epochs (L=46, R=44)
  S007: 90 epochs (L=46, R=44)
  S008: 90 epochs (L=44, R=46)
  S009: 90 epochs (L=47, R=43)
  S010: 90 epochs (L=48, R=42)
  S011: 90 epochs (L=45, R=45)
  S012: 90 epochs (L=45, R=45)
  S013: 90 epochs (L=47, R=43)
  S014: 90 epochs (L=44, R=46)
  S015: 90 epochs (L=45, R=45)
  S016: 90 epochs (L=45, R=45)
  S017: 90 epochs (L=46, R=44)
  S018: 90 epochs (L=44, R=46)
  S019: 90 epochs (L=47, R=43)
  S020: 90 epochs (L=44, R=46)
  S021: 90 epochs (L=48, R=42)
  S022: 90 epochs (L=45, R=45)
  S023: 90 epochs (L=44, R=46)
  S024: 90 epochs (L=45, R=45)
  S025: 90 epochs (L=45, R=45)
  S026: 90 epochs (L=47, R=43)
  S027: 90 epochs (L=46, R=44)
  S028: 90 epochs (L=45, R=45)
  S029: 90 epochs (L=45, R=45)
  S030: 90 epochs (L=46, R=44)
  S031: 90 epochs (L=45, R=45)
  S032: 90 ep

## Preprocessing

In [3]:
def bandpass_filter(X_trials, sfreq=SFREQ, low=BANDPASS_LOW, high=BANDPASS_HIGH, order=5):
    nyquist = sfreq / 2
    b, a = butter(order, [low / nyquist, high / nyquist], btype='band')
    X_filtered = np.zeros_like(X_trials)
    for i, trial in enumerate(X_trials):
        X_filtered[i] = filtfilt(b, a, trial, axis=1)
    return X_filtered


def channel_wise_zscore(X_trials):
    N, C, T = X_trials.shape
    X_flat = X_trials.reshape(N, C * T)
    mean = X_flat.mean(axis=0, keepdims=True)
    std = X_flat.std(axis=0, keepdims=True) + 1e-8
    return ((X_flat - mean) / std).reshape(N, C, T)


def channel_wise_zscore_from_train(X_train, X_test):
    N_train, C, T = X_train.shape
    X_train_flat = X_train.reshape(N_train, C * T)
    mean = X_train_flat.mean(axis=0, keepdims=True)
    std = X_train_flat.std(axis=0, keepdims=True) + 1e-8
    X_train_norm = (X_train.reshape(N_train, C * T) - mean) / std
    X_test_norm = (X_test.reshape(X_test.shape[0], C * T) - mean) / std
    return X_train_norm.reshape(N_train, C, T), X_test_norm.reshape(X_test.shape[0], C, T)


def preprocess_per_subject(X_trials, y_trials, sfreq=SFREQ):
    X = bandpass_filter(X_trials, sfreq=sfreq)
    X = channel_wise_zscore(X)
    return X, y_trials


preprocessed_data = {}
for sid, (X, y) in subjects_data.items():
    X_pp, y_pp = preprocess_per_subject(X, y)
    preprocessed_data[sid] = (X_pp, y_pp)
print(f'Preprocessed {len(preprocessed_data)} subjects')

Preprocessed 109 subjects


## EEGTCNet Model

In [4]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=64, input_time_len=641, emb_size=40,
                 pool_kernel=15, pool_stride=4):
        super().__init__()
        self.conv_temp = nn.Conv2d(1, 40, kernel_size=(1, 25), stride=(1, 1), padding=(0, 0))
        self.conv_spat = nn.Conv2d(40, 40, kernel_size=(in_channels, 1), stride=(1, 1), padding=(0, 0))
        self.bn = nn.BatchNorm2d(40)
        self.act = nn.ELU()
        self.pool = nn.AvgPool2d(kernel_size=(1, pool_kernel), stride=(1, pool_stride))
        self.drop = nn.Dropout(p=0.5)
        self.proj = nn.Conv2d(40, emb_size, kernel_size=(1, 1), stride=(1, 1))
        self.input_time_len = input_time_len
        self.pool_kernel = pool_kernel
        self.pool_stride = pool_stride

    def compute_seq_len(self):
        W1 = self.input_time_len - 25 + 1
        Wp = max(1, (W1 - self.pool_kernel) // self.pool_stride + 1) if W1 >= self.pool_kernel else 1
        return Wp

    def forward(self, x):
        x = self.conv_temp(x)
        x = self.conv_spat(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.pool(x)
        x = self.drop(x)
        x = self.proj(x)
        x = x.squeeze(2).permute(0, 2, 1)
        return x


class StandardTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_hidden=128, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim), nn.Dropout(dropout))

    def forward(self, x):
        shortcut = x
        x = self.ln1(x)
        x = shortcut + self.attn(x, x, x)[0]
        shortcut = x
        x = self.ln2(x)
        return shortcut + self.mlp(x)


class StandardTransformer(nn.Module):
    def __init__(self, dim=64, num_layers=3, num_heads=4, mlp_hidden=128, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            StandardTransformerBlock(dim=dim, num_heads=num_heads,
                                     mlp_hidden=mlp_hidden, dropout=dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x.mean(dim=1)


class TokenReconstructionHead(nn.Module):
    def __init__(self, tf_dim=64, emb_size=40):
        super().__init__()
        self.proj = nn.Linear(tf_dim, emb_size)

    def forward(self, features):
        return self.proj(features)


class EEGTCNetV4(nn.Module):
    def __init__(self, in_channels=64, input_time_len=641, emb_size=40,
                 tf_dim=64, num_tf_layers=3, num_heads=4,
                 mlp_hidden=128, num_classes=2,
                 pool_kernel=15, pool_stride=4):
        super().__init__()
        self.patch_embedding = PatchEmbedding(
            in_channels=in_channels, input_time_len=input_time_len,
            emb_size=emb_size, pool_kernel=pool_kernel, pool_stride=pool_stride)
        self.seq_len = self.patch_embedding.compute_seq_len()
        self.emb_size = emb_size
        self.tf_dim = tf_dim
        self.embedding_projection = nn.Linear(emb_size, tf_dim)
        self.pos_encoding = nn.Parameter(torch.zeros(1, self.seq_len, tf_dim))
        nn.init.trunc_normal_(self.pos_encoding, std=0.02)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, tf_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.transformer = StandardTransformer(
            dim=tf_dim, num_layers=num_tf_layers, num_heads=num_heads,
            mlp_hidden=mlp_hidden, dropout=0.1)
        self.classification_head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(tf_dim, num_classes))

    def forward(self, x, mask=None, return_features=False):
        x = self.patch_embedding(x)
        x = self.embedding_projection(x)
        x = x + self.pos_encoding[:, :x.size(1), :]
        if mask is not None:
            x[mask] = self.mask_token
        features = self.transformer(x)
        logits = self.classification_head(features)
        if return_features:
            return logits, features, x
        return logits


MODEL_KWARGS = dict(in_channels=N_CHANNELS, input_time_len=EPOCH_SAMPLES,
                    num_classes=N_CLASSES, pool_kernel=POOL_KERNEL, pool_stride=POOL_STRIDE)

model = EEGTCNetV4(**MODEL_KWARGS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'k15_s4: {model.seq_len} tokens, {n_params:,} params')
dummy = torch.randn(2, 1, N_CHANNELS, EPOCH_SAMPLES).to(device)
print(f'Output shape: {model(dummy).shape}')

k15_s4: 151 tokens, 218,226 params
Output shape: torch.Size([2, 2])


## Training

In [9]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode='acc'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best = 0.0 if mode == 'acc' else float('inf')
        self.best_state = None

    def __call__(self, metric, model):
        improved = (metric > self.best + self.min_delta) if self.mode == 'acc' else (metric < self.best - self.min_delta)
        if improved:
            self.best = metric if self.mode == 'acc' else min(metric, self.best)
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)
        return model


def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(X_test).to(device)
        if X_t.dim() == 3:
            X_t = X_t.unsqueeze(1)
        outputs = model(X_t)
        probs = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
    return accuracy_score(y_test, predicted.cpu().numpy()), predicted.cpu().numpy(), probs.cpu().numpy()


def pretrain_masked_reconstruction(model, X_all, epochs=50, lr=1e-4,
                                    mask_ratio=0.15, batch_size=72):
    model = model.to(device)
    recon_head = TokenReconstructionHead(tf_dim=model.tf_dim, emb_size=model.emb_size).to(device)
    all_params = list(model.parameters()) + list(recon_head.parameters())
    optimizer = torch.optim.Adam(all_params, lr=lr)

    if X_all.ndim == 3:
        X_all = X_all[:, np.newaxis, :, :]
    X_t = torch.FloatTensor(X_all).to(device)
    n_tokens = model.seq_len

    for epoch in range(epochs):
        model.train()
        recon_head.train()
        perm = torch.randperm(len(X_t))
        epoch_loss = 0
        n_batches = 0
        for i in range(0, len(X_t), batch_size):
            X_b = X_t[perm[i:i+batch_size]]
            optimizer.zero_grad()
            with torch.no_grad():
                patch_emb = model.patch_embedding(X_b)
                patch_emb_proj = model.embedding_projection(patch_emb)
            B = X_b.size(0)
            mask = torch.rand(B, n_tokens) < mask_ratio
            mask = mask.to(device)
            logits, features, _ = model(X_b, mask=mask, return_features=True)
            recon = recon_head(features)
            target = patch_emb_proj.mean(dim=1)
            loss = F.mse_loss(recon, target[:, :model.emb_size])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        if (epoch + 1) % 10 == 0:
            print(f'  Pretrain epoch {epoch+1}/{epochs} | Loss: {epoch_loss/n_batches:.4f}')
    return model, recon_head


def train_model(model, X_train, y_train, X_val, y_val,
                epochs=MAX_EPOCHS, lr=LR, batch_size=BATCH_SIZE,
                patience=PATIENCE, weight_decay=WEIGHT_DECAY):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    if X_train.ndim == 3:
        X_train = X_train[:, np.newaxis, :, :]
        X_val = X_val[:, np.newaxis, :, :] if X_val.ndim == 3 else X_val

    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.LongTensor(y_train).to(device)
    X_val_t = torch.FloatTensor(X_val).to(device)
    y_val_t = torch.LongTensor(y_val).to(device)

    early_stopper = EarlyStopping(patience=patience, mode='acc')
    best_state = None

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_train_t))
        epoch_loss, correct, total = 0, 0, 0

        for i in range(0, len(X_train_t), batch_size):
            X_b = X_train_t[perm[i:i+batch_size]]
            y_b = y_train_t[perm[i:i+batch_size]]
            optimizer.zero_grad()
            out = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            _, pred = torch.max(out, 1)
            total += y_b.size(0)
            correct += (pred == y_b).sum().item()

        train_acc = correct / total

        model.eval()
        with torch.no_grad():
            val_out = model(X_val_t)
            val_loss = criterion(val_out, y_val_t).item()
            _, val_pred = torch.max(val_out, 1)
            val_acc = (val_pred == y_val_t).sum().item() / len(y_val_t)

        scheduler.step(val_loss)

        if early_stopper(val_acc, model):
            model = early_stopper.restore(model)
            break

    return model, best_state


def finetune_head_only(pretrained_state, model_kwargs, X_calib, y_calib, X_test, y_test,
                       epochs=FT_EPOCHS, lr=FT_LR):
    model = EEGTCNetV4(**model_kwargs)
    model.load_state_dict(pretrained_state, strict=False)
    model = model.to(device)

    for name, param in model.named_parameters():
        param.requires_grad = 'classification_head' in name

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    if X_calib.ndim == 3:
        X_calib = X_calib[:, np.newaxis, :, :]
        X_test = X_test[:, np.newaxis, :, :] if X_test.ndim == 3 else X_test

    X_cal_t = torch.FloatTensor(X_calib).to(device)
    y_cal_t = torch.LongTensor(y_calib).to(device)
    X_test_t = torch.FloatTensor(X_test).to(device)

    best_test_acc = 0.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(X_cal_t)
        loss = criterion(out, y_cal_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            test_out = model(X_test_t)
            _, test_pred = torch.max(test_out, 1)
            test_acc = (test_pred.cpu().numpy() == y_test).mean()

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    acc, _, _ = evaluate_model(model, X_test, y_test)
    return acc


print('Training utilities ready.')

Training utilities ready.


## LOSO Evaluation

In [10]:
os.makedirs(SAVE_DIR, exist_ok=True)
all_sids = sorted(preprocessed_data.keys())
loso_results = []
loso_test_data = {}

results_file = os.path.join(SAVE_DIR, 'loso_results.npy')
if os.path.exists(results_file):
    saved = np.load(results_file, allow_pickle=True).item()
    loso_results = saved['results']
    print(f'Resumed {len(loso_results)} completed subjects')
else:
    print(f'Running LOSO on {len(all_sids)} subjects...')

completed_sids = {r['subject'] for r in loso_results}

for test_sid in all_sids:
    if test_sid in completed_sids:
        continue

    train_sids = [s for s in all_sids if s != test_sid]
    X_train_arr = np.concatenate([preprocessed_data[s][0] for s in train_sids])
    y_train_arr = np.concatenate([preprocessed_data[s][1] for s in train_sids])
    X_test_arr, y_test_arr = preprocessed_data[test_sid]

    X_train_norm, X_test_norm = channel_wise_zscore_from_train(X_train_arr, X_test_arr)
    loso_test_data[test_sid] = {'X_test_norm': X_test_norm, 'y_test': y_test_arr}

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_norm, y_train_arr, test_size=0.1, stratify=y_train_arr, random_state=42)

    # Self-supervised pretraining
    model_pt = EEGTCNetV4(**MODEL_KWARGS)
    model_pt, _ = pretrain_masked_reconstruction(model_pt, X_train_norm,
                                                   epochs=PRETRAIN_EPOCHS, lr=PRETRAIN_LR,
                                                   mask_ratio=PRETRAIN_MASK_RATIO)
    pretrain_state = {k: v.clone() for k, v in model_pt.state_dict().items()}

    # Supervised from pretrained weights
    model_loso = EEGTCNetV4(**MODEL_KWARGS)
    model_loso.load_state_dict(pretrain_state, strict=False)
    nn.init.xavier_uniform_(model_loso.classification_head[1].weight)
    nn.init.zeros_(model_loso.classification_head[1].bias)

    model_loso, _ = train_model(model_loso, X_tr, y_tr, X_val, y_val)
    acc, _, _ = evaluate_model(model_loso, X_test_norm, y_test_arr)
    loso_results.append({'subject': test_sid, 'accuracy': acc})
    print(f'S{test_sid:03d}: {acc*100:.1f}% ({len(loso_results)}/{len(all_sids)})')

    # Save checkpoint
    ckpt_path = os.path.join(SAVE_DIR, f'loso_subject_{test_sid:03d}.pt')
    torch.save(model_loso.state_dict(), ckpt_path)

    # Incremental save
    np.save(results_file, {'results': loso_results})

accs = np.array([r['accuracy'] for r in loso_results])
print(f'\nLOSO: {accs.mean()*100:.1f}% +/- {accs.std()*100:.1f}% ({len(accs)} subjects)')
print(f'Range: {accs.min()*100:.1f}% - {accs.max()*100:.1f}%')

Resumed 109 completed subjects

LOSO: 80.8% +/- 9.4% (109 subjects)
Range: 55.6% - 98.9%


## Fine-Tuning

In [11]:
print(f'Fine-tuning at {CALIBRATION_FRACTIONS}')

# Rebuild test data for any subjects missing from loso_test_data (e.g. resumed results)
for test_sid in all_sids:
    if test_sid not in loso_test_data:
        train_sids = [s for s in all_sids if s != test_sid]
        X_train_arr = np.concatenate([preprocessed_data[s][0] for s in train_sids])
        y_train_arr = np.concatenate([preprocessed_data[s][1] for s in train_sids])
        X_test_arr, y_test_arr = preprocessed_data[test_sid]
        X_train_norm, X_test_norm = channel_wise_zscore_from_train(X_train_arr, X_test_arr)
        loso_test_data[test_sid] = {'X_test_norm': X_test_norm, 'y_test': y_test_arr}
print(f'Test data ready for {len(loso_test_data)} subjects')

ft_results = {f'{int(f*100)}%': [] for f in CALIBRATION_FRACTIONS}

for test_sid in all_sids:
    ckpt_path = os.path.join(SAVE_DIR, f'loso_subject_{test_sid:03d}.pt')
    if not os.path.exists(ckpt_path):
        continue

    pretrained_state = torch.load(ckpt_path, map_location=device)
    X_test_norm = loso_test_data[test_sid]['X_test_norm']
    y_test_raw = loso_test_data[test_sid]['y_test']

    zero_shot_acc = [r['accuracy'] for r in loso_results if r['subject'] == test_sid][0]

    for frac in CALIBRATION_FRACTIONS:
        frac_key = f'{int(frac*100)}%'
        try:
            X_cal, X_te, y_cal, y_te = train_test_split(
                X_test_norm, y_test_raw, train_size=frac, stratify=y_test_raw, random_state=42)
        except ValueError:
            continue

        acc_ft = finetune_head_only(pretrained_state, MODEL_KWARGS, X_cal, y_cal, X_te, y_te)
        ft_results[frac_key].append({
            'subject': test_sid, 'ft_accuracy': acc_ft,
            'zero_shot': zero_shot_acc, 'improvement': acc_ft - zero_shot_acc,
            'n_calib': len(y_cal)
        })

# Print summary
accs_loso = np.array([r['accuracy'] for r in loso_results])
print(f'\nZero-shot LOSO: {accs_loso.mean()*100:.1f}%')
for frac in CALIBRATION_FRACTIONS:
    frac_key = f'{int(frac*100)}%'
    results = ft_results[frac_key]
    if results:
        accs_ft = np.array([r['ft_accuracy'] for r in results])
        imps = np.array([r['improvement'] for r in results])
        print(f'  head_only @ {frac_key}: {accs_ft.mean()*100:.1f}% (delta={imps.mean()*100:+.1f}pp)')

Fine-tuning at [0.05, 0.1, 0.2, 0.5]
Test data ready for 109 subjects

Zero-shot LOSO: 80.8%
  head_only @ 5%: 81.8% (delta=+1.4pp)
  head_only @ 10%: 81.8% (delta=+1.4pp)
  head_only @ 20%: 81.7% (delta=+1.3pp)
  head_only @ 50%: 82.2% (delta=+1.8pp)
